# Option Pricing, Greeks, and Delta Hedging


In [2]:
import numpy as np
import pandas as pd

## American Option Pricing with a Binomial Tree

### – Price ATM American Call and Put

In [5]:
def binomial_american_option(S0, K, r, sigma, T, n_steps, option_type='call'):
    """
    Price an American option using the Cox-Ross-Rubinstein binomial tree.
    
    Parameters:
    ----------
    S0 :initial stock price
    K :strike price
    r :risk-free rate (annualized)
    sigma :volatility (annualized)
    T :time to maturity (in years)
    n_steps :number of steps in the tree
    option_type:'call' or 'put'
    
    Returns:
    --------
    price :option price at t=0
    tree_values :ption values at each node (for Delta computation)
    tree_prices :stock prices at each node
    """
    dt = T / n_steps
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    discount = np.exp(-r * dt)
    
    # Build stock price tree (each row = one time step)
    tree_prices = np.zeros((n_steps + 1, n_steps + 1))
    for i in range(n_steps + 1):
        for j in range(i + 1):
            tree_prices[j, i] = S0 * (u ** (i - j)) * (d ** j)
    
    # Final payoffs
    tree_values = np.zeros_like(tree_prices)
    if option_type == 'call':
        tree_values[:, n_steps] = np.maximum(tree_prices[:, n_steps] - K, 0)
    else:  # put
        tree_values[:, n_steps] = np.maximum(K - tree_prices[:, n_steps], 0)
    
    # Backward induction with early exercise check
    for i in range(n_steps - 1, -1, -1):
        for j in range(i + 1):
            hold_value = discount * (p * tree_values[j, i + 1] + (1 - p) * tree_values[j + 1, i + 1])
            if option_type == 'call':
                exercise_value = max(tree_prices[j, i] - K, 0)
            else:
                exercise_value = max(K - tree_prices[j, i], 0)
            tree_values[j, i] = max(hold_value, exercise_value)
    
    return tree_values[0, 0], tree_values, tree_prices

In [9]:
n_steps = 100

am_call_price, am_call_tree, am_call_prices = binomial_american_option(
    S0_step1, K_atm, r_step1, sigma_step1, T_step1, n_steps, 'call'
)
am_put_price, am_put_tree, am_put_prices = binomial_american_option(
    S0_step1, K_atm, r_step1, sigma_step1, T_step1, n_steps, 'put'
)

q8_results = pd.DataFrame({
    'Option': ['American Call', 'American Put'],
    'Price': [round(am_call_price, 2), round(am_put_price, 2)]
})
print(q8_results.to_string(index=False))

       Option  Price
American Call   4.61
 American Put   3.47


### – Delta for American Call and Put

In [12]:
def binomial_delta(tree_values, tree_prices, n_steps):
    """
    Compute Delta at t=0 from binomial tree.
    Delta = (V_up - V_down) / (S_up - S_down)
    """
    V_up = tree_values[0, 1]   
    V_down = tree_values[1, 1]  
    S_up = tree_prices[0, 1]
    S_down = tree_prices[1, 1]
    return (V_up - V_down) / (S_up - S_down)

delta_call = binomial_delta(am_call_tree, am_call_prices, n_steps)
delta_put = binomial_delta(am_put_tree, am_put_prices, n_steps)

q9_results = pd.DataFrame({
    'Option': ['American Call', 'American Put'],
    'Delta': [round(delta_call, 4), round(delta_put, 4)]
})
print(q9_results.to_string(index=False))
print("\nCall Delta - Put Delta = {:.4f}".format(delta_call - delta_put))

       Option   Delta
American Call  0.5693
 American Put -0.4498

Call Delta - Put Delta = 1.0191


### – Vega (Sensitivity to Volatility)

In [15]:
sigma_low = 0.20
sigma_high = 0.25

am_call_low, _, _ = binomial_american_option(S0_step1, K_atm, r_step1, sigma_low, T_step1, n_steps, 'call')
am_call_high, _, _ = binomial_american_option(S0_step1, K_atm, r_step1, sigma_high, T_step1, n_steps, 'call')
am_put_low, _, _ = binomial_american_option(S0_step1, K_atm, r_step1, sigma_low, T_step1, n_steps, 'put')
am_put_high, _, _ = binomial_american_option(S0_step1, K_atm, r_step1, sigma_high, T_step1, n_steps, 'put')

vega_call = (am_call_high - am_call_low) / 0.05
vega_put = (am_put_high - am_put_low) / 0.05

q10_results = pd.DataFrame({
    'Option': ['American Call', 'American Put'],
    'Price vol20%': [round(am_call_low, 2), round(am_put_low, 2)],
    'Price vol25%': [round(am_call_high, 2), round(am_put_high, 2)],
    'Change ': [round(am_call_high - am_call_low, 2), round(am_put_high - am_put_low, 2)],
    'Vega': [round(vega_call, 2), round(vega_put, 2)]
})
print("Q10a: Sensitivity to 5% Volatility Increase (20% → 25%)")
print(q10_results.to_string(index=False))

       Option  Price vol20%  Price vol25%  Change   Vega
American Call          4.61          5.59     0.98 19.62
 American Put          3.47          4.45     0.98 19.57


## European Option Pricing with a Trinomial Tree

### – European Calls at 5 Strikes

In [16]:
def trinomial_european_option(S0, K, r, sigma, T, n_steps, option_type='call'):
    """
    Price a European option using a trinomial tree.
    
    Parameters:
    ----------
    S0, K, r, sigma, T : as in binomial
    n_steps :number of steps
    option_type : 'call' or 'put'
    
    Returns:
    --------
    price
    """
    dt = T / n_steps
    u = np.exp(sigma * np.sqrt(2 * dt))
    d = 1 / u
    # Risk-neutral probabilities 
    exp_rhs = np.exp(sigma * np.sqrt(dt / 2))
    exp_rls = np.exp(-sigma * np.sqrt(dt / 2))
    denom = exp_rhs - exp_rls
    pu = ((np.exp(r * dt / 2) - exp_rls) / denom) ** 2
    pd = ((exp_rhs - np.exp(r * dt / 2)) / denom) ** 2
    pm = 1 - pu - pd
    discount = np.exp(-r * dt)
    
    n_nodes = 2 * n_steps + 1
    tree_prices = np.full((n_nodes, n_steps + 1), np.nan)
    tree_values = np.zeros((n_nodes, n_steps + 1))
    
    for i in range(n_steps + 1):
        for j in range(2 * i + 1):
            net = j - i  # net up moves
            tree_prices[j, i] = S0 * (u ** net)
    
    # Terminal payoffs
    for j in range(2 * n_steps + 1):
        S_T = tree_prices[j, n_steps]
        if not np.isnan(S_T):
            if option_type == 'call':
                tree_values[j, n_steps] = max(S_T - K, 0)
            else:
                tree_values[j, n_steps] = max(K - S_T, 0)
    
    # Backward induction (European: no early exercise)
    for i in range(n_steps - 1, -1, -1):
        for j in range(2 * i + 1):
            hold = discount * (pu * tree_values[j + 2, i + 1] + 
                              pm * tree_values[j + 1, i + 1] + 
                              pd * tree_values[j, i + 1])
            tree_values[j, i] = hold
    
    return tree_values[0, 0]

In [20]:
strikes = [110, 105, 100, 95, 90]
moneyness = [round(k/S0_step1*100, 0) for k in strikes]
n_trinomial = 100

eu_call_prices = [round(trinomial_european_option(S0_step1, K, r_step1, sigma_step1, T_step1, n_trinomial, 'call'), 2) 
                  for K in strikes]

q15_results = pd.DataFrame({
    'Moneyness (K/S0)': moneyness,
    'European Call Price': eu_call_prices
})
print(q15_results.to_string(index=False))

 Moneyness (K/S0)  European Call Price
            110.0                 1.19
            105.0                 2.48
            100.0                 4.61
             95.0                 7.72
             90.0                11.67


### – European Puts at 5 Strikes

In [23]:
eu_put_prices = [round(trinomial_european_option(S0_step1, K, r_step1, sigma_step1, T_step1, n_trinomial, 'put'), 2) 
                 for K in strikes]

q16_results = pd.DataFrame({
    'Moneyness (K/S0 %)': moneyness,
    'European Put Price': eu_put_prices
})
print(q16_results.to_string(index=False))
# Put-call parity: C - P = S0 - K*exp(-r*T)
print("\nPut-call parity check (European):")
disc = np.exp(-r_step1 * T_step1)
for i, K in enumerate(strikes):
    lhs = eu_call_prices[i] - eu_put_prices[i]
    rhs = S0_K * disc
    print("  K={}: C-P={:.2f}, S0-Ke^(-rT)={:.2f}, diff={:.4f}".format(K, lhs, rhs, abs(lhs-rhs)))

 Moneyness (K/S0 %)  European Put Price
              110.0                9.83
              105.0                6.18
              100.0                3.37
               95.0                1.54
               90.0                0.55

Put-call parity check (European):
  K=110: C-P=-8.64, S0-Ke^(-rT)=-8.63, diff=0.0064
  K=105: C-P=-3.70, S0-Ke^(-rT)=-3.70, diff=0.0043
  K=100: C-P=1.24, S0-Ke^(-rT)=1.24, diff=0.0022
  K=95: C-P=6.18, S0-Ke^(-rT)=6.18, diff=0.0001
  K=90: C-P=11.12, S0-Ke^(-rT)=11.12, diff=0.0020


## Delta Hedging with a Three-Step Binomial Tree

### – European Put, 3-Step Binomial

In [25]:
S0_step3 = 180
r_step3 = 0.02
sigma_step3 = 0.25
T_step3 = 6/12
K_step3 = 182

def binomial_european_option(S0, K, r, sigma, T, n_steps, option_type='call'):
    """Same as American, but no early exercise"""
    dt = T / n_steps
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    discount = np.exp(-r * dt)
    tree_prices = np.zeros((n_steps + 1, n_steps + 1))
    tree_values = np.zeros((n_steps + 1, n_steps + 1))
    for i in range(n_steps + 1):
        for j in range(i + 1):
            tree_prices[j, i] = S0 * (u ** (i - j)) * (d ** j)
    if option_type == 'call':
        tree_values[:, n_steps] = np.maximum(tree_prices[:, n_steps] - K, 0)
    else:
        tree_values[:, n_steps] = np.maximum(K - tree_prices[:, n_steps], 0)
    for i in range(n_steps - 1, -1, -1):
        for j in range(i + 1):
            tree_values[j, i] = discount * (p * tree_values[j, i + 1] + (1 - p) * tree_values[j + 1, i + 1])
    return tree_values[0, 0], tree_values, tree_prices

In [31]:
n_q25 = 3
eu_put_3step, eu_put_3step_vals, eu_put_3step_stock = binomial_european_option(
    S0_step3, K_step3, r_step3, sigma_step3, T_step3, n_q25, 'put'
)
eu_put_3step_deltas = np.full((n_q25+1, n_q25+1), np.nan)
for i in range(n_q25):
    for j in range(i+1):
        eu_put_3step_deltas[j, i] = (eu_put_3step_vals[j, i+1] - eu_put_3step_vals[j+1, i+1]) / (eu_put_3step_stock[j, i+1] - eu_put_3step_stock[j+1, i+1])
for j in range(n_q25+1):
    eu_put_3step_deltas[j, n_q25] = -1.0 if eu_put_3step_stock[j, n_q25] < K_step3 else 0.0


print("\nStock tree (columns = steps 0..3):")
for j in range(n_q25+1):
    row = [round(eu_put_3step_stock[j, i], 2) for i in range(n_q25+1) if j <= i]
    print("  Node {}:".format(j), row)


Stock tree (columns = steps 0..3):
  Node 0: [np.float64(180.0), np.float64(199.34), np.float64(220.76), np.float64(244.48)]
  Node 1: [np.float64(162.54), np.float64(180.0), np.float64(199.34)]
  Node 2: [np.float64(146.77), np.float64(162.54)]
  Node 3: [np.float64(132.52)]


In [32]:

path_q25 = [0, 1, 2, 3]
dt_q25 = T_step3 / n_q25
premium_q25 = eu_put_3step
B_q25 = premium_q25 - eu_put_3step_deltas[0, 0] * eu_put_3step_stock[0, 0]
rows_q25 = [{'Step': 0, 'S': round(eu_put_3step_stock[0,0], 2), 'Delta': round(eu_put_3step_deltas[0,0], 4), 'Shares': round(eu_put_3step_deltas[0,0], 4), 'Cash': round(B_q25, 2)}]
delta_prev = eu_put_3step_deltas[0, 0]
for i in range(1, n_q25 + 1):
    j = path_q25[i]
    S = eu_put_3step_stock[j, i]
    delta_curr = eu_put_3step_deltas[j, i]
    B_q25 = B_q25 * np.exp(r_step3 * dt_q25) + (delta_prev - delta_curr) * S
    delta_prev = delta_curr
    rows_q25.append({'Step': i, 'S': round(S, 2), 'Delta': round(delta_curr, 4), 'Shares': round(delta_curr, 4), 'Cash': round(B_q25, 2)})
q25_table = pd.DataFrame(rows_q25)

print(q25_table.to_string(index=False))

 Step      S   Delta  Shares   Cash
    0 180.00 -0.4726 -0.4726  98.88
    1 162.54 -0.7447 -0.7447 143.45
    2 146.77 -1.0000 -1.0000 181.39
    3 132.52 -1.0000 -1.0000 182.00


## Answer Table

In [28]:
# Answer Table – consolidated results
df_answers = pd.DataFrame([
    ('', 'American ATM (S0=100, K=100)', f'Call ${round(am_call_price, 2)}, Put ${round(am_put_price, 2)}'),
    ('', 'Deltas at t=0', f'Call {round(delta_call, 4)}, Put {round(delta_put, 4)}'),
    ('', 'Vega (20%→25%)', f'Both +${round(am_call_high - am_call_low, 2)}'),
    ('', 'European Calls (K=110..90)', ', '.join(str(p) for p in eu_call_prices)),
    ('', 'European Puts (K=110..90)', ', '.join(str(p) for p in eu_put_prices)),
    ('', 'European Put 3-step (K=182)', f'${round(eu_put_3step, 2)} (cash account: see Q25b above)'),
], columns=['Question', 'Description', 'Result'])
print(df_answers.to_string(index=False))

Question                  Description                                Result
      Q8 American ATM (S0=100, K=100)                 Call $4.61, Put $3.47
      Q9                Deltas at t=0              Call 0.5693, Put -0.4498
     Q10               Vega (20%→25%)                           Both +$0.98
     Q15   European Calls (K=110..90)         1.19, 2.48, 4.61, 7.72, 11.67
     Q16    European Puts (K=110..90)          9.83, 6.18, 3.37, 1.54, 0.55
     Q25  European Put 3-step (K=182) $13.82 (cash account: see Q25b above)
